In [7]:
!pip install scikit-learn scipy numpy pandas micromlgen

In [8]:
import os
import numpy as np
import pandas as pd
from scipy.stats import skew, kurtosis
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from micromlgen import port

In [9]:
import os
files = os.listdir('/content')
print(files)

['.config', 'Night 4.csv', 'Night 5.csv', 'Night 8.csv', 'Night 1.csv', 'Night 7.csv', 'Night 2.csv', 'Night 3.csv', 'Night 6.csv', 'No Movement.csv', 'sample_data']


In [10]:
df = pd.read_csv('/content/Night 2.csv')
print(df.shape)
print(df.head())
print(df.columns.tolist())


(47551, 5)
             timestamp     aX   aY    aZ  label
0  2026-05-26 04:57:00  16572 -516 -1720  light
1  2026-05-26 04:57:00  16660 -548 -1784  light
2  2026-05-26 04:57:01  16660 -528 -1788  light
3  2026-05-26 04:57:01  16764 -484 -1868  light
4  2026-05-26 04:57:02  16592 -508 -1812  light
['timestamp', 'aX', 'aY', 'aZ', 'label']


In [11]:
NIGHTS_DIR = '/content'
COLUMN_AXES = ['aX', 'aY', 'aZ']
COLUMN_LABEL = 'label'

TEST_NIGHTS = ['Night 1.csv', 'Night 2.csv']

SAMPLE_RATE_HZ = 2
WINDOW_SIZE = SAMPLE_RATE_HZ * 30   # 60 rows = 30 seconds
WINDOW_STRIDE = SAMPLE_RATE_HZ * 10  # 20 rows = 10 seconds

print(f"Window size: {WINDOW_SIZE} rows")
print(f"Window stride: {WINDOW_STRIDE} rows")

Window size: 60 rows
Window stride: 20 rows


In [12]:
def compute_features(window):
    features = []
    for col_idx in range(3):
        x = window[:, col_idx].astype(float)
        features.append(np.mean(x))
        features.append(np.min(x))
        features.append(np.max(x))
        features.append(np.sqrt(np.mean(x ** 2)))
        features.append(np.std(x))
        features.append(float(skew(x)))
        features.append(float(kurtosis(x)))
    return np.array(features, dtype=np.float32)

print("compute_features defined")

compute_features defined


In [13]:
def extract_windows(df):
    X, y = [], []
    data = df[COLUMN_AXES].values
    labels = df[COLUMN_LABEL]

    for start in range(0, len(data) - WINDOW_SIZE + 1, WINDOW_STRIDE):
        window_data = data[start : start + WINDOW_SIZE]
        window_labels = labels.iloc[start : start + WINDOW_SIZE]
        X.append(compute_features(window_data))
        y.append(window_labels.value_counts().idxmax())

    return np.array(X), np.array(y)

print("extract_windows defined")

extract_windows defined


In [14]:
X_train_list, y_train_list = [], []
X_test_list,  y_test_list  = [], []

all_files = sorted([f for f in os.listdir(NIGHTS_DIR) if f.endswith('.csv')])

for filename in all_files:
    filepath = os.path.join(NIGHTS_DIR, filename)
    df = pd.read_csv(filepath)
    df = df[df[COLUMN_LABEL].str.lower() != 'awake'].reset_index(drop=True)

    if len(df) < WINDOW_SIZE:
        print(f"SKIP  | {filename} — only {len(df)} rows after filtering")
        continue

    X, y = extract_windows(df)
    counts = dict(zip(*np.unique(y, return_counts=True)))

    if filename in TEST_NIGHTS:
        X_test_list.append(X)
        y_test_list.append(y)
        print(f"TEST  | {filename}: {len(y)} windows — {counts}")
    else:
        X_train_list.append(X)
        y_train_list.append(y)
        print(f"TRAIN | {filename}: {len(y)} windows — {counts}")

X_train = np.vstack(X_train_list)
y_train = np.concatenate(y_train_list)
X_test  = np.vstack(X_test_list)
y_test  = np.concatenate(y_test_list)

print(f"\nTraining: {X_train.shape[0]} windows")
print(f"Test:     {X_test.shape[0]} windows")

TEST  | Night 1.csv: 2629 windows — {np.str_('deep'): np.int64(653), np.str_('light'): np.int64(1976)}
TEST  | Night 2.csv: 2375 windows — {np.str_('deep'): np.int64(288), np.str_('light'): np.int64(2087)}
TRAIN | Night 3.csv: 3364 windows — {np.str_('deep'): np.int64(947), np.str_('light'): np.int64(2417)}
TRAIN | Night 4.csv: 2794 windows — {np.str_('deep'): np.int64(870), np.str_('light'): np.int64(1924)}
TRAIN | Night 5.csv: 2644 windows — {np.str_('deep'): np.int64(366), np.str_('light'): np.int64(2278)}
TRAIN | Night 6.csv: 1388 windows — {np.str_('deep'): np.int64(492), np.str_('light'): np.int64(896)}
TRAIN | Night 7.csv: 1606 windows — {np.str_('deep'): np.int64(234), np.str_('light'): np.int64(1372)}
TRAIN | Night 8.csv: 2770 windows — {np.str_('deep'): np.int64(652), np.str_('light'): np.int64(2118)}
TRAIN | No Movement.csv: 1584 windows — {np.str_('deep'): np.int64(1584)}

Training: 16150 windows
Test:     5004 windows


In [15]:
clf = RandomForestClassifier(
    n_estimators=20,
    max_depth=10,
    class_weight='balanced',
    random_state=42
)
clf.fit(X_train, y_train)
print("Done training")
print(f"Classes: {clf.classes_}")

Done training
Classes: ['deep' 'light']


In [16]:
y_pred = clf.predict(X_test)
print(classification_report(y_test, y_pred, digits=3))

              precision    recall  f1-score   support

        deep      0.154     0.189     0.170       941
       light      0.802     0.760     0.780      4063

    accuracy                          0.652      5004
   macro avg      0.478     0.474     0.475      5004
weighted avg      0.680     0.652     0.665      5004



In [17]:
importances = clf.feature_importances_
feature_names = []
for axis in ['aX', 'aY', 'aZ']:
    for stat in ['mean', 'min', 'max', 'rms', 'std', 'skew', 'kurtosis']:
        feature_names.append(f"{axis}_{stat}")

for name, imp in sorted(zip(feature_names, importances), key=lambda x: -x[1]):
    print(f"{name}: {imp:.3f}")

aZ_rms: 0.166
aZ_min: 0.148
aZ_mean: 0.123
aY_rms: 0.111
aY_mean: 0.103
aZ_max: 0.063
aY_min: 0.055
aY_max: 0.052
aX_rms: 0.040
aX_mean: 0.029
aX_min: 0.018
aX_max: 0.014
aZ_std: 0.010
aY_std: 0.010
aY_kurtosis: 0.009
aZ_kurtosis: 0.009
aY_skew: 0.009
aX_kurtosis: 0.008
aX_skew: 0.008
aZ_skew: 0.008
aX_std: 0.007


In [18]:
def compute_features(window):
    features = []
    for col_idx in range(3):
        x = window[:, col_idx].astype(float)
        x = x - np.mean(x)  # remove gravity/position offset
        features.append(np.mean(x))
        features.append(np.min(x))
        features.append(np.max(x))
        features.append(np.sqrt(np.mean(x ** 2)))
        features.append(np.std(x))
        features.append(float(skew(x)))
        features.append(float(kurtosis(x)))
    return np.array(features, dtype=np.float32)

print("Updated compute_features defined")

Updated compute_features defined


In [19]:
# Re-extract windows with gravity removed
X_train_list, y_train_list = [], []
X_test_list,  y_test_list  = [], []

for filename in all_files:
    filepath = os.path.join(NIGHTS_DIR, filename)
    df = pd.read_csv(filepath)
    df = df[df[COLUMN_LABEL].str.lower() != 'awake'].reset_index(drop=True)

    if len(df) < WINDOW_SIZE:
        continue

    X, y = extract_windows(df)

    if filename in TEST_NIGHTS:
        X_test_list.append(X)
        y_test_list.append(y)
    else:
        X_train_list.append(X)
        y_train_list.append(y)

X_train = np.vstack(X_train_list)
y_train = np.concatenate(y_train_list)
X_test  = np.vstack(X_test_list)
y_test  = np.concatenate(y_test_list)

# Retrain
clf = RandomForestClassifier(n_estimators=20, max_depth=10, class_weight='balanced', random_state=42)
clf.fit(X_train, y_train)

# Evaluate
y_pred = clf.predict(X_test)
print(classification_report(y_test, y_pred, digits=3))

              precision    recall  f1-score   support

        deep      0.206     0.125     0.156       941
       light      0.814     0.888     0.849      4063

    accuracy                          0.744      5004
   macro avg      0.510     0.507     0.503      5004
weighted avg      0.700     0.744     0.719      5004



In [20]:
clf = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    class_weight='balanced',
    random_state=42
)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)
print(classification_report(y_test, y_pred, digits=3))

              precision    recall  f1-score   support

        deep      0.236     0.104     0.144       941
       light      0.816     0.922     0.866      4063

    accuracy                          0.768      5004
   macro avg      0.526     0.513     0.505      5004
weighted avg      0.707     0.768     0.730      5004



In [21]:
# Original features — no gravity removal
def compute_features(window):
    features = []
    for col_idx in range(3):
        x = window[:, col_idx].astype(float)
        features.append(np.mean(x))
        features.append(np.min(x))
        features.append(np.max(x))
        features.append(np.sqrt(np.mean(x ** 2)))
        features.append(np.std(x))
        features.append(float(skew(x)))
        features.append(float(kurtosis(x)))
    return np.array(features, dtype=np.float32)

# Re-extract windows
X_train_list, y_train_list = [], []
X_test_list,  y_test_list  = [], []

for filename in all_files:
    filepath = os.path.join(NIGHTS_DIR, filename)
    df = pd.read_csv(filepath)
    df = df[df[COLUMN_LABEL].str.lower() != 'awake'].reset_index(drop=True)
    if len(df) < WINDOW_SIZE:
        continue
    X, y = extract_windows(df)
    if filename in TEST_NIGHTS:
        X_test_list.append(X)
        y_test_list.append(y)
    else:
        X_train_list.append(X)
        y_train_list.append(y)

X_train = np.vstack(X_train_list)
y_train = np.concatenate(y_train_list)
X_test  = np.vstack(X_test_list)
y_test  = np.concatenate(y_test_list)

# Retrain
clf = RandomForestClassifier(n_estimators=20, max_depth=10, class_weight='balanced', random_state=42)
clf.fit(X_train, y_train)

# Confirm results
y_pred = clf.predict(X_test)
print(classification_report(y_test, y_pred, digits=3))
print(f"Classes: {clf.classes_}")

              precision    recall  f1-score   support

        deep      0.154     0.189     0.170       941
       light      0.802     0.760     0.780      4063

    accuracy                          0.652      5004
   macro avg      0.478     0.474     0.475      5004
weighted avg      0.680     0.652     0.665      5004

Classes: ['deep' 'light']


In [22]:
from micromlgen import port

c_code = port(clf)

with open('/content/sleep_classifier.h', 'w') as f:
    f.write(c_code)

print(f"Exported successfully")
print(f"File size: {len(c_code)} characters")

Exported successfully
File size: 1393470 characters


In [23]:
from google.colab import files
files.download('/content/sleep_classifier.h')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>